# MAPED on MPS

Load five synthetic acquisitions, align detector and scan shifts, merge, and inspect the result. Run `python make_data.py synthetic-tilts` once first. See [README](README.md) for the required development environment. No experimental data is used.

The input adapts the [PR #256 validation example](https://github.com/electronmicroscopy/quantem/pull/256): five 64 × 64 scans, each containing 64 × 64 diffraction patterns. All coordinates are pixels.

In [ ]:
from pathlib import Path

from quantem.gpu import io
from quantem.diffraction import MAPED
from quantem.widget import Show4DSTEM

folder = Path("synthetic-tilts")
device = "mps"

## Load

Select the compute device here. Each acquisition stays on that device throughout alignment and merging. The synthetic files require no detector-mask or hot-pixel correction.

In [ ]:
files = io.discover(folder)
tilts = io.load(
    files, device=device,
    apply_mask=False, hot_pixel_correction="none",
)
maped = MAPED.from_datasets(tilts)
maped.show_tilts()

## Align

Find each direct beam, align the diffraction patterns, then align the scan images. The table reports the applied **(row, col)** corrections in pixels; a common translation is arbitrary.

In [ ]:
maped.find_beam_center(show=False)
maped.align_diffraction(show=False)

In [ ]:
maped.align_real_space(iterations=80, show=False)
maped.report()

## Merge and explore

Keep float32 values and crop to the scan region covered by every tilt. Drag the virtual detector to change the image; click the scan image to inspect a diffraction pattern. The browser renders the viewer; numerical queries use the dataset’s selected device.

In [ ]:
merged = maped.merge(dtype="float32", crop=True, show=False)
viewer = Show4DSTEM(merged, device=device)
viewer

## Save and reopen

HDF5 retains the float32 values and alignment metadata. These files are new outputs; the input files remain unchanged.

In [ ]:
output = Path("merged_mps_master.h5")
io.save(output, merged, backend="mps")
reopened = io.load(output, device=device)
reopened

For the same workflow in one call: `MAPED.from_datasets(tilts).run(iterations=80, dtype="float32", show=False)`. Exact float32 output is limited to 4 GiB; see the README before using a full experimental scan.